In [149]:
import os
from pathlib import Path
import librosa
import soundfile as sf
import numpy as np

REAL_INPUT_DIR = "data/realAudio" 
FAKE_INPUT_DIR = "data/fakeAudio"

CLEAN_REAL_DIR = "data/cleanReal" 
CLEAN_FAKE_DIR = "data/cleanFake"

TARGET_SR = 16000
TARGET_DURATION = 2.0


def standardizeAudio(in_path, out_path, sr=TARGET_SR, duration=TARGET_DURATION):
    # Load WAV
    y, _ = librosa.load(in_path, sr=sr, mono=True)
    target_len = int(sr * duration)
    # If too short → pad to center
    if len(y) < target_len:
        pad_total = target_len - len(y)
        pad_left = pad_total // 2
        pad_right = pad_total - pad_left
        y = np.pad(y, (pad_left, pad_right))
    else:
        # Extract the middle 2 seconds
        mid = len(y) // 2
        half = target_len // 2
        start = max(0, mid - half)
        end = start + target_len
        y = y[start:end]
    # Ensure output folder exists
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    # Save standardized WAV
    sf.write(out_path, y, sr)
    print(f"Saved: {out_path}")


def processFolder(input_dir, output_dir):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    audio_files = list(input_dir.glob("*.wav"))
    if not audio_files:
        print(f"No .wav files found in {input_dir}")
        return
    for f in audio_files:
        out_name = f.stem + "Clean.wav"
        out_path = output_dir / out_name
        standardizeAudio(str(f), str(out_path))


def main():
    print("Processing REAL WAV audio")
    processFolder(REAL_INPUT_DIR, CLEAN_REAL_DIR)

    print("\nProcessing FAKE WAV audio")
    processFolder(FAKE_INPUT_DIR, CLEAN_FAKE_DIR)


if __name__ == "__main__":
    main()

Processing REAL WAV audio
Saved: data\cleanReal\audio1Clean.wav
Saved: data\cleanReal\audio10Clean.wav
Saved: data\cleanReal\audio11Clean.wav
Saved: data\cleanReal\audio12Clean.wav
Saved: data\cleanReal\audio13Clean.wav
Saved: data\cleanReal\audio14Clean.wav
Saved: data\cleanReal\audio15Clean.wav
Saved: data\cleanReal\audio16Clean.wav
Saved: data\cleanReal\audio17Clean.wav
Saved: data\cleanReal\audio18Clean.wav
Saved: data\cleanReal\audio19Clean.wav
Saved: data\cleanReal\audio2Clean.wav
Saved: data\cleanReal\audio20Clean.wav
Saved: data\cleanReal\audio3Clean.wav
Saved: data\cleanReal\audio4Clean.wav
Saved: data\cleanReal\audio5Clean.wav
Saved: data\cleanReal\audio6Clean.wav
Saved: data\cleanReal\audio7Clean.wav
Saved: data\cleanReal\audio8Clean.wav
Saved: data\cleanReal\audio9Clean.wav

Processing FAKE WAV audio
Saved: data\cleanFake\audio0Clean.wav
Saved: data\cleanFake\audio1Clean.wav
Saved: data\cleanFake\audio10Clean.wav
Saved: data\cleanFake\audio11Clean.wav
Saved: data\cleanFake\

In [150]:
import os
from pathlib import Path

import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt

REAL_INPUT_DIR = "data/cleanReal"
FAKE_INPUT_DIR = "data/cleanFake"

REAL_OUTPUT_DIR = "data/realSpect"
FAKE_OUTPUT_DIR = "data/fakeSpect"

SAMPLE_RATE = 16000


def create_spectrogram(wav_path, png_path, sr=SAMPLE_RATE):
    # Load audio
    y, sr = librosa.load(wav_path, sr=sr, mono=True)
    # Mel spectrogram
    S = librosa.feature.melspectrogram(
        y=y, sr=sr,
        n_fft=2048,
        hop_length=512,
        n_mels=128
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    # Plot
    plt.figure(figsize=(8, 3))
    librosa.display.specshow(
        S_db, sr=sr, hop_length=512,
        x_axis="time", y_axis="mel"
    )
    plt.colorbar(label="dB")
    plt.title(Path(wav_path).name)
    plt.tight_layout()
    # Save png
    os.makedirs(os.path.dirname(png_path), exist_ok=True)
    plt.savefig(png_path, dpi=150)
    plt.close()

    print(f"Saved: {png_path}")


def processFolder(input_dir, output_dir):
    """Create spectrogram PNGs for all WAV files in a folder."""
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    wav_files = sorted(input_dir.glob("*.wav"))
    if not wav_files:
        print(f"No .wav files found in {input_dir}")
        return

    for wav_file in wav_files:
        out_name = wav_file.stem + "_spect.png"
        out_path = output_dir / out_name
        create_spectrogram(str(wav_file), str(out_path))


def main():
    print("Processing cleaned REAL audio")
    processFolder(REAL_INPUT_DIR, REAL_OUTPUT_DIR)

    print("\nProcessing cleaned FAKE audio")
    processFolder(FAKE_INPUT_DIR, FAKE_OUTPUT_DIR)


if __name__ == "__main__":
    main()

Processing cleaned REAL audio
Saved: data\realSpect\audio10Clean_spect.png
Saved: data\realSpect\audio11Clean_spect.png
Saved: data\realSpect\audio12Clean_spect.png
Saved: data\realSpect\audio13Clean_spect.png
Saved: data\realSpect\audio14Clean_spect.png
Saved: data\realSpect\audio15Clean_spect.png
Saved: data\realSpect\audio16Clean_spect.png
Saved: data\realSpect\audio17Clean_spect.png
Saved: data\realSpect\audio18Clean_spect.png
Saved: data\realSpect\audio19Clean_spect.png
Saved: data\realSpect\audio1Clean_spect.png
Saved: data\realSpect\audio20Clean_spect.png
Saved: data\realSpect\audio2Clean_spect.png
Saved: data\realSpect\audio3Clean_spect.png
Saved: data\realSpect\audio4Clean_spect.png
Saved: data\realSpect\audio5Clean_spect.png
Saved: data\realSpect\audio6Clean_spect.png
Saved: data\realSpect\audio7Clean_spect.png
Saved: data\realSpect\audio8Clean_spect.png
Saved: data\realSpect\audio9Clean_spect.png

Processing cleaned FAKE audio
Saved: data\fakeSpect\audio0Clean_spect.png
Save

In [151]:
import os
import glob
import numpy as np
import pandas as pd
import librosa

def extractFingerprint(filepath, sr=16000, n_mfcc=13):
    # Load audio 
    y, sr = librosa.load(filepath, sr=sr, mono=True)
    features = {}

    # MFCC
    mfcc = librosa.feature.mfcc(
        y=y, sr=sr, n_mfcc=n_mfcc,
        dct_type=2, norm='ortho'   # for stable reproducibility
    )

    for i in range(n_mfcc):
        features[f"mfcc_{i+1}_mean"] = float(mfcc[i].mean())
        features[f"mfcc_{i+1}_std"] = float(mfcc[i].std())

    # Spectral features
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.95)
    flatness = librosa.feature.spectral_flatness(y=y)

    features["spect_centroid_mean"] = float(centroid.mean())
    features["spect_centroid_std"] = float(centroid.std())

    features["spect_bandwidth_mean"] = float(bandwidth.mean())
    features["spect_bandwidth_std"] = float(bandwidth.std())

    features["spect_rolloff95_mean"] = float(rolloff.mean())
    features["spect_rolloff95_std"] = float(rolloff.std())

    features["spect_flatness_mean"] = float(flatness.mean())
    features["spect_flatness_std"] = float(flatness.std())

    # Temporal / energy
    zcr = librosa.feature.zero_crossing_rate(y)
    rms = librosa.feature.rms(y=y)
    flux = librosa.onset.onset_strength(y=y, sr=sr)

    features["zcr_mean"] = float(zcr.mean())
    features["zcr_std"] = float(zcr.std())

    features["rms_mean"] = float(rms.mean())
    features["rms_std"] = float(rms.std())

    features["spec_flux_mean"] = float(flux.mean())
    features["spec_flux_std"] = float(flux.std())

    return features


def processFolders(real_dir, fake_dir):
    rows = []
    # Real
    for filepath in glob.glob(os.path.join(real_dir, "*.wav")):
        print(f"Processing (real): {filepath}")
        feats = extractFingerprint(filepath)
        feats["label"] = "real"
        feats["file"] = os.path.basename(filepath)
        rows.append(feats)

    # Fake
    for filepath in glob.glob(os.path.join(fake_dir, "*.wav")):
        print(f"Processing (deepfake): {filepath}")
        feats = extractFingerprint(filepath)
        feats["label"] = "deepfake"
        feats["file"] = os.path.basename(filepath)
        rows.append(feats)

    return pd.DataFrame(rows)


if __name__ == "__main__":
    REAL_DIR = "data/cleanReal"
    FAKE_DIR = "data/cleanFake"
    OUTPUT_CSV = "data/spectralFingerprints.csv"

    df = processFolders(REAL_DIR, FAKE_DIR)
    print("Extracted features shape:", df.shape)
    print(df.head())

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved features to {OUTPUT_CSV}")

Processing (real): data/cleanReal\audio10Clean.wav
Processing (real): data/cleanReal\audio11Clean.wav
Processing (real): data/cleanReal\audio12Clean.wav
Processing (real): data/cleanReal\audio13Clean.wav
Processing (real): data/cleanReal\audio14Clean.wav
Processing (real): data/cleanReal\audio15Clean.wav
Processing (real): data/cleanReal\audio16Clean.wav
Processing (real): data/cleanReal\audio17Clean.wav
Processing (real): data/cleanReal\audio18Clean.wav
Processing (real): data/cleanReal\audio19Clean.wav
Processing (real): data/cleanReal\audio1Clean.wav
Processing (real): data/cleanReal\audio20Clean.wav
Processing (real): data/cleanReal\audio2Clean.wav
Processing (real): data/cleanReal\audio3Clean.wav
Processing (real): data/cleanReal\audio4Clean.wav
Processing (real): data/cleanReal\audio5Clean.wav
Processing (real): data/cleanReal\audio6Clean.wav
Processing (real): data/cleanReal\audio7Clean.wav
Processing (real): data/cleanReal\audio8Clean.wav
Processing (real): data/cleanReal\audio

This is the OPTICS algorithm and it's effect on the code. I tried to increase min_sample, but that just dropped the accuracy .75. At it's current tuning it will give you an accuracy of 90.91%.

In [152]:
"""from sklearn.cluster import OPTICS
import numpy as np

CSV_PATH = "data/spectralFingerprints.csv"
df = pd.read_csv(CSV_PATH)
X = df.drop(columns=["label", "file"])
y = df["label"]
clustering = OPTICS(min_samples=2, metric='euclidean', max_eps=20000).fit(X)
print(clustering.labels_)
df['outlier']= clustering.labels_
#print(X.outlier)
indices_to_drop = df[df['outlier'] == -1].index
df_new = df.drop(indices_to_drop)
print(df_new.outlier)
df_final = df_new.drop(columns='outlier')"""


'from sklearn.cluster import OPTICS\nimport numpy as np\n\nCSV_PATH = "data/spectralFingerprints.csv"\ndf = pd.read_csv(CSV_PATH)\nX = df.drop(columns=["label", "file"])\ny = df["label"]\nclustering = OPTICS(min_samples=2, metric=\'euclidean\', max_eps=20000).fit(X)\nprint(clustering.labels_)\ndf[\'outlier\']= clustering.labels_\n#print(X.outlier)\nindices_to_drop = df[df[\'outlier\'] == -1].index\ndf_new = df.drop(indices_to_drop)\nprint(df_new.outlier)\ndf_final = df_new.drop(columns=\'outlier\')'

This is the Z-score outlier detection. This was a bit less use full since I had to manually look and take out the indencies that were giving us trouble [7, 28, 35, 38] for 100% accuracy.

In [153]:
"""import numpy as np
from scipy import stats

CSV_PATH = "data/spectralFingerprints.csv"
df = pd.read_csv(CSV_PATH)
X = df.drop(columns=["label", "file"])
y = df["label"]
z_scores = np.abs(stats.zscore(X))
outliers = np.where(z_scores > 2) # Example threshold
#print(f"Outliers based on Z-score: {np.array(X)[outliers]}")
#print(z_scores)
print(outliers)
stuff_to_drop = df.index[[7, 28, 35, 38]]
df_final = df.drop(stuff_to_drop)"""

'import numpy as np\nfrom scipy import stats\n\nCSV_PATH = "data/spectralFingerprints.csv"\ndf = pd.read_csv(CSV_PATH)\nX = df.drop(columns=["label", "file"])\ny = df["label"]\nz_scores = np.abs(stats.zscore(X))\noutliers = np.where(z_scores > 2) # Example threshold\n#print(f"Outliers based on Z-score: {np.array(X)[outliers]}")\n#print(z_scores)\nprint(outliers)\nstuff_to_drop = df.index[[7, 28, 35, 38]]\ndf_final = df.drop(stuff_to_drop)'

This is the isolation forest that takes the x part of the dataset and gives us the outliers. By removing said outliers, we were able to get 100% accuracy

In [154]:
from sklearn.ensemble import IsolationForest
CSV_PATH = "data/spectralFingerprints.csv"
df = pd.read_csv(CSV_PATH)
X = df.drop(columns=["label", "file"])
y = df["label"]
clf = IsolationForest(random_state=0).fit_predict(X)
print(clf)
df['outlier']= clf
#print(X.outlier)
indices_to_drop = df[df['outlier'] == -1].index
df_new = df.drop(indices_to_drop)
print(df_new.outlier)
df_final = df_new.drop(columns='outlier')

[ 1  1  1  1  1  1  1 -1  1 -1  1  1  1  1  1  1  1  1  1  1  1  1  1  1
  1  1  1  1 -1  1  1  1  1  1  1 -1  1  1 -1 -1  1  1  1  1  1  1  1 -1
  1  1]
0     1
1     1
2     1
3     1
4     1
5     1
6     1
8     1
10    1
11    1
12    1
13    1
14    1
15    1
16    1
17    1
18    1
19    1
20    1
21    1
22    1
23    1
24    1
25    1
26    1
27    1
29    1
30    1
31    1
32    1
33    1
34    1
36    1
37    1
40    1
41    1
42    1
43    1
44    1
45    1
46    1
48    1
49    1
Name: outlier, dtype: int32


If you want to change this back to normal, delete the final of the df for X and y.

In [155]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import Normalizer
from sklearn.preprocessing import MinMaxScaler

# Path to your CSV from the feature-extraction script
CSV_PATH = "data/spectralFingerprints.csv"

def main():
    # Load data
    df = pd.read_csv(CSV_PATH)

    # Drop non-feature columns
    X = df_final.drop(columns=["label", "file"])
    y = df_final["label"]

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        random_state=42,
        stratify=y
    )

    # Pipeline: standardize features -> KNN
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=5))
    ])

    # Train
    clf.fit(X_train, y_train)

    print("\nCross validation score:")
    print(cross_val_score(clf, X_train, y_train))
    print()

    # Evaluate
    acc = clf.score(X_test, y_test)
    print(f"Accuracy: {acc:.4f}\n")

    y_pred = clf.predict(X_test)

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    main()


Cross validation score:
[1.         0.83333333 0.83333333 1.         0.83333333]

Accuracy: 1.0000

Confusion Matrix:
[[8 0]
 [0 5]]

Classification Report:
              precision    recall  f1-score   support

    deepfake       1.00      1.00      1.00         8
        real       1.00      1.00      1.00         5

    accuracy                           1.00        13
   macro avg       1.00      1.00      1.00        13
weighted avg       1.00      1.00      1.00        13



In [156]:
import os
import glob
import numpy as np
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import Normalizer
from sklearn.preprocessing import MinMaxScaler

def load_spectrogram(filepath, size=(128, 128)):
    """
    Load a PNG spectrogram, convert to grayscale, resize,
    normalize, and flatten to 1D.
    """
    img = Image.open(filepath).convert("L")  # grayscale
    img = img.resize(size)                  # enforce fixed size
    arr = np.array(img, dtype=np.float32)

    # normalize to [0,1]
    arr /= 255.0

    # flatten into a 1D feature vector
    return arr.flatten()


def load_spectrogram_dataset(real_dir, fake_dir, img_size=(128,128)):
    X, y, files = [], [], []

    # Real spectrograms
    for filepath in glob.glob(os.path.join(real_dir, "*.png")):
        feat = load_spectrogram(filepath, size=img_size)
        X.append(feat)
        y.append("real")
        files.append(os.path.basename(filepath))

    # Fake spectrograms
    for filepath in glob.glob(os.path.join(fake_dir, "*.png")):
        feat = load_spectrogram(filepath, size=img_size)
        X.append(feat)
        y.append("deepfake")
        files.append(os.path.basename(filepath))

    X = np.vstack(X)
    y = np.array(y)
    files = np.array(files)

    print("Dataset loaded.")
    print("Feature matrix shape:", X.shape)
    print("Class counts:")
    print("  real:", np.sum(y=="real"))
    print("  deepfake:", np.sum(y=="deepfake"))

    return X, y, files


if __name__ == "__main__":
    REAL_SPECT_DIR = "data/realSpect"
    FAKE_SPECT_DIR = "data/fakeSpect"

    # Load data
    X, y, files = load_spectrogram_dataset(REAL_SPECT_DIR, FAKE_SPECT_DIR)

    # Split dataset
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.3,
        random_state=42,
        stratify=y
    )

    # KNN pipeline with StandardScaler
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=5))
    ])

    clf.fit(X_train, y_train)
    
    print("\nCross validation score:")
    print(cross_val_score(clf, X_train, y_train))
    print()

    # Evaluate
    acc = clf.score(X_test, y_test)
    print(f"\nAccuracy: {acc:.4f}")

    y_pred = clf.predict(X_test)

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred, labels=["real", "deepfake"]))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, labels=["real", "deepfake"]))

Dataset loaded.
Feature matrix shape: (50, 16384)
Class counts:
  real: 20
  deepfake: 30

Cross validation score:
[0.42857143 0.57142857 1.         0.28571429 0.57142857]


Accuracy: 0.8000

Confusion Matrix:
[[5 1]
 [2 7]]

Classification Report:
              precision    recall  f1-score   support

        real       0.71      0.83      0.77         6
    deepfake       0.88      0.78      0.82         9

    accuracy                           0.80        15
   macro avg       0.79      0.81      0.80        15
weighted avg       0.81      0.80      0.80        15

